<a href="https://colab.research.google.com/github/cactus1386/NationalCard-ImageProccessing/blob/main/trainOCR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from PIL import Image
import os
import yaml

In [3]:
class CNNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1):
        super(CNNBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, padding=padding)
        self.bn = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = nn.functional.relu(x)
        return x

In [4]:
class CRNN(nn.Module):
    def __init__(self, n_channels, num_classes, map2seq_in_dim, map2seq_out_dim, rnn_dim):
        super(CRNN, self).__init__()
        self.cnn = nn.ModuleDict({
            '0': CNNBlock(n_channels, 64),
            '2': CNNBlock(64, 128),
            '4': CNNBlock(128, 256),
            '5': CNNBlock(256, 256),
            '7': CNNBlock(256, 512),
            '8': CNNBlock(512, 512),
            '10': CNNBlock(512, 512, kernel_size=3, padding=1)
        })
        self.map2seq = nn.Linear(map2seq_in_dim, map2seq_out_dim)
        self.rnn1 = nn.LSTM(map2seq_out_dim, rnn_dim, bidirectional=True, batch_first=True)
        self.rnn2 = nn.LSTM(rnn_dim * 2, rnn_dim, bidirectional=True, batch_first=True)
        self.classifier = nn.Linear(rnn_dim * 2, num_classes)

    def forward(self, x):
        for key in sorted(self.cnn.keys(), key=lambda k: int(k)):
            x = self.cnn[key](x)
            if key in ['0', '2']:
                x = nn.functional.max_pool2d(x, 2)
            elif key in ['5', '8']:
                x = nn.functional.max_pool2d(x, (2, 1))
        batch, channels, height, width = x.size()
        x = x.permute(0, 3, 1, 2).reshape(batch, width, -1)
        x = self.map2seq(x)
        x, _ = self.rnn1(x)
        x, _ = self.rnn2(x)
        x = self.classifier(x)
        return x

In [5]:
def preprocess_image(image_path):
    with open("/content/drive/MyDrive/crnn-fa-printed-96-long/preprocessor/image_processor_config.yaml", "r") as f:
        config = yaml.safe_load(f)

    transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((32, 384)),
        transforms.Lambda(lambda x: x.transpose(Image.FLIP_LEFT_RIGHT) if config["mirror"] else x),
        transforms.ToTensor(),
        transforms.Normalize(mean=config["mean"], std=config["std"]),
        transforms.Lambda(lambda x: x * config["rescale"])
    ])
    image = Image.open(image_path).convert("L")
    return transform(image)

In [6]:
class NewOCRDataset(Dataset):
    def __init__(self, image_dir):
        self.image_dir = image_dir
        self.image_paths = [os.path.join(image_dir, fname) for fname in os.listdir(image_dir) if fname.endswith(('.png', '.jpg', '.jpeg'))]

        with open("/content/drive/MyDrive/crnn-fa-printed-96-long/model_config.yaml", "r") as f:
            config = yaml.safe_load(f)
        self.char2id = {char: idx for idx, char in config["id2label"].items()}

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = preprocess_image(img_path)

        label = os.path.splitext(os.path.basename(img_path))[0]
        label = format_label(label)

        label_ids = [self.char2id.get(c, 0) for c in label]
        return image, torch.tensor(label_ids, dtype=torch.long), len(label_ids)

In [7]:
def collate_fn(batch):
    images, targets, target_lengths = zip(*batch)
    images = torch.stack(images, dim=0)
    targets = [t.clone().detach() for t in targets]
    target_lengths = torch.tensor(target_lengths, dtype=torch.long)
    return images, targets, target_lengths

In [8]:
def train_model(model, train_dataloader, val_dataloader, num_epochs=5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    criterion = nn.CTCLoss(blank=0, zero_infinity=True)
    optimizer = optim.Adam(model.parameters(), lr=0.0001)

    for epoch in range(num_epochs):
        # آموزش
        model.train()
        total_train_loss = 0
        for batch_idx, (images, targets, target_lengths) in enumerate(train_dataloader):
            images = images.to(device)
            targets = torch.cat(targets).to(device)
            target_lengths = target_lengths.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            outputs = outputs.log_softmax(2)
            outputs = outputs.permute(1, 0, 2)
            input_lengths = torch.full((images.size(0),), outputs.size(0), dtype=torch.long).to(device)
            loss = criterion(outputs, targets, input_lengths, target_lengths)

            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()
            print(f"Epoch [{epoch+1}/{num_epochs}], Batch [{batch_idx}], Train Loss: {loss.item():.4f}")

        avg_train_loss = total_train_loss / len(train_dataloader)
        print(f"Epoch [{epoch+1}/{num_epochs}] completed, Avg Train Loss: {avg_train_loss:.4f}")

        # ارزیابی روی validation
        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for images, targets, target_lengths in val_dataloader:
                images = images.to(device)
                targets = torch.cat(targets).to(device)
                target_lengths = target_lengths.to(device)

                outputs = model(images)
                outputs = outputs.log_softmax(2)
                outputs = outputs.permute(1, 0, 2)
                input_lengths = torch.full((images.size(0),), outputs.size(0), dtype=torch.long).to(device)
                loss = criterion(outputs, targets, input_lengths, target_lengths)

                total_val_loss += loss.item()

        avg_val_loss = total_val_loss / len(val_dataloader)
        print(f"Epoch [{epoch+1}/{num_epochs}] completed, Avg Val Loss: {avg_val_loss:.4f}")

    torch.save(model.state_dict(), "finetuned.pt")
    print("مدل ذخیره شد: finetuned.pt")

In [9]:
def format_label(label):
    if label.isdigit():
        if len(label) == 8:
            year = label[:4]
            month = label[4:6]
            day = label[6:]
            return f"{year}/{month}/{day}"
        elif len(label) == 10:
            return label
    return label

In [10]:
from torch.utils.data import random_split

# مسیر فولدر تصاویر
image_dir = "/content/drive/MyDrive/ImageGenerate/images"

# ایجاد دیتاست
dataset = NewOCRDataset(image_dir)

# محاسبه اندازه‌های train/test/validation
total_size = len(dataset)
train_size = int(0.7 * total_size)  # 70% برای train
val_size = int(0.15 * total_size)   # 15% برای validation
test_size = total_size - train_size - val_size  # بقیه برای test

# تقسیم دیتاست
train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])

# ایجاد DataLoader برای هر بخش
train_dataloader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)
val_dataloader = DataLoader(val_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)
test_dataloader = DataLoader(test_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)

In [11]:
with open("/content/drive/MyDrive/crnn-fa-printed-96-long/model_config.yaml", "r") as f:
    config = yaml.safe_load(f)

model = CRNN(
    n_channels=config["n_channels"],
    map2seq_in_dim=config["map2seq_in_dim"],
    map2seq_out_dim=config["map2seq_out_dim"],
    rnn_dim=config["rnn_dim"],
    num_classes=len(config["id2label"])
)

model.load_state_dict(torch.load("/content/drive/MyDrive/crnn-fa-printed-96-long/model.pt", map_location="cpu"))
model.eval()

# آموزش مدل
train_model(model, train_dataloader, val_dataloader, num_epochs=30)

# ارزیابی روی دیتاست تست (اختیاری)
model.eval()
total_test_loss = 0
with torch.no_grad():
    for images, targets, target_lengths in test_dataloader:
        targets = torch.cat(targets)

        outputs = model(images)
        outputs = outputs.log_softmax(2)
        outputs = outputs.permute(1, 0, 2)
        input_lengths = torch.full((images.size(0),), outputs.size(0), dtype=torch.long)
        loss = criterion(outputs, targets, input_lengths, target_lengths)

        total_test_loss += loss.item()

avg_test_loss = total_test_loss / len(test_dataloader)
print(f"Test Loss: {avg_test_loss:.4f}")

Epoch [1/30], Batch [0], Train Loss: 8.5565
Epoch [1/30], Batch [1], Train Loss: 6.6508
Epoch [1/30], Batch [2], Train Loss: 3.5074
Epoch [1/30], Batch [3], Train Loss: 4.7723
Epoch [1/30], Batch [4], Train Loss: 4.2740
Epoch [1/30], Batch [5], Train Loss: 3.7287
Epoch [1/30], Batch [6], Train Loss: 3.0705
Epoch [1/30], Batch [7], Train Loss: 3.6553
Epoch [1/30], Batch [8], Train Loss: 4.2066
Epoch [1/30], Batch [9], Train Loss: 2.3950
Epoch [1/30], Batch [10], Train Loss: 2.6833
Epoch [1/30], Batch [11], Train Loss: 5.3238
Epoch [1/30], Batch [12], Train Loss: 2.2958
Epoch [1/30], Batch [13], Train Loss: 2.8748
Epoch [1/30], Batch [14], Train Loss: 2.3520
Epoch [1/30], Batch [15], Train Loss: 1.6478
Epoch [1/30], Batch [16], Train Loss: 1.4304
Epoch [1/30], Batch [17], Train Loss: 6.9601
Epoch [1/30], Batch [18], Train Loss: 2.7729
Epoch [1/30], Batch [19], Train Loss: 2.7716
Epoch [1/30], Batch [20], Train Loss: 1.3511
Epoch [1/30], Batch [21], Train Loss: 3.0393
Epoch [1/30], Batch 

KeyboardInterrupt: 